In [2]:
# =============================================================
# CELL 1: Setup and ticker selection
# =============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path("/Users/yashaswiaryan/Projects/WOLF")
PRICE_DIR = BASE_DIR / "full_history"
NEWS_FILE = BASE_DIR / "All_external.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# --- TICKER SELECTION ---
# Criteria: good news coverage + good price history + recognizable names
# Mix of sectors for diversity
SELECTED_TICKERS = [
    # High news coverage from your chart + well-known
    "GILD",   # Biotech (Gilead) — top news coverage
    "HD",     # Retail (Home Depot)
    "FDX",    # Logistics (FedEx)
    "FCX",    # Mining (Freeport-McMoRan)
    "BLK",    # Finance (BlackRock)
    "EA",     # Gaming (Electronic Arts)
    "TXN",    # Semiconductor (Texas Instruments)
    "OXY",    # Energy (Occidental Petroleum)
    "AXP",    # Finance (American Express)
    "HAL",    # Energy (Halliburton)
    # Adding some mega-caps if they exist in your data
    "AAPL",   # Tech
    "MSFT",   # Tech
    "AMZN",   # Tech/Retail
    "TSLA",   # Auto/Tech
    "GOOGL",  # Tech
]
# Verify which of these actually exist in both datasets
available_price = {f.stem for f in PRICE_DIR.glob("*.csv")}
print(f"Selected tickers in price data: {[t for t in SELECTED_TICKERS if t in available_price]}")
print(f"Missing from price data: {[t for t in SELECTED_TICKERS if t not in available_price]}")


Selected tickers in price data: ['GILD', 'HD', 'FDX', 'FCX', 'BLK', 'EA', 'TXN', 'OXY', 'AXP', 'HAL', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'GOOGL']
Missing from price data: []


In [11]:
pip install pyarrow

  Using cached pyarrow-23.0.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp313-cp313-macosx_12_0_arm64.whl (34.2 MB)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
# =============================================================
# CELL 2: Load and clean stock price data
# =============================================================

def clean_price_data(ticker: str) -> pd.DataFrame:
    """Load and clean a single ticker's price CSV."""
    filepath = PRICE_DIR / f"{ticker}.csv"
    if not filepath.exists():
        print(f"  ⚠️ {ticker}.csv not found — skipping")
        return None
    
    df = pd.read_csv(filepath)
    
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    
    # Parse dates
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    
    # Sort chronologically (your data is newest-first)
    df = df.sort_values("date").reset_index(drop=True)
    
    # Add ticker column
    df["ticker"] = ticker
    
    # --- Handle missing values ---
    numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    # Ensure columns exist
    for col in numeric_cols:
        if col not in df.columns:
            print(f"  ⚠️ {ticker}: missing column '{col}'")
    
    # Drop rows where close price is missing (essential)
    df = df.dropna(subset=["close"])
    
    # Forward-fill minor gaps in other columns (1-2 days)
    df[numeric_cols] = df[numeric_cols].ffill(limit=2)
    
    # --- Handle outliers ---
    # Flag extreme daily returns (>50% in a day = likely stock split or error)
    df["daily_return"] = df["close"].pct_change()
    extreme = df["daily_return"].abs() > 0.50
    if extreme.sum() > 0:
        print(f"  ⚠️ {ticker}: {extreme.sum()} extreme daily returns (>50%) — likely splits")
    
    # Remove the helper column
    df = df.drop(columns=["daily_return"])
    
    # --- Compute returns and log returns ---
    df["return_1d"] = df["adj_close"].pct_change()
    df["log_return"] = np.log(df["adj_close"] / df["adj_close"].shift(1))
    
    # Drop first row (NaN from pct_change)
    df = df.iloc[1:].reset_index(drop=True)
    
    return df

# Process all selected tickers
print("Cleaning stock price data...")
price_dfs = []
for ticker in SELECTED_TICKERS:
    print(f"  Processing {ticker}...")
    df = clean_price_data(ticker)
    if df is not None:
        price_dfs.append(df)
        print(f"    ✅ {len(df)} rows | {df['date'].min().date()} → {df['date'].max().date()}")

all_prices = pd.concat(price_dfs, ignore_index=True)
print(f"\n📊 Combined price data: {len(all_prices):,} rows, {all_prices['ticker'].nunique()} tickers")
print(f"Date range: {all_prices['date'].min().date()} → {all_prices['date'].max().date()}")

# Save checkpoint
# Save as CSV (parquet has compatibility issues with Python 3.13)
all_prices.to_csv(PROCESSED_DIR / "prices_clean.csv", index=False)
print(f"💾 Saved to {PROCESSED_DIR / 'prices_clean.csv'}")

Cleaning stock price data...
  Processing GILD...
    ✅ 8043 rows | 1992-01-23 → 2023-12-28
  Processing HD...
    ✅ 9769 rows | 1981-09-23 → 2020-06-19
  Processing FDX...
    ✅ 11527 rows | 1978-04-13 → 2023-12-28
  Processing FCX...
    ✅ 6226 rows | 1995-07-11 → 2020-04-01
  Processing BLK...
    ✅ 5157 rows | 1999-10-04 → 2020-04-01
  Processing EA...
    ✅ 8634 rows | 1989-09-21 → 2023-12-28
  Processing TXN...
    ✅ 13005 rows | 1972-06-02 → 2023-12-28
  Processing OXY...
  ⚠️ OXY: 1 extreme daily returns (>50%) — likely splits
    ✅ 10586 rows | 1982-01-04 → 2023-12-28
  Processing AXP...
    ✅ 13004 rows | 1972-06-02 → 2023-12-28
  Processing HAL...
    ✅ 13005 rows | 1972-06-02 → 2023-12-28
  Processing AAPL...
  ⚠️ AAPL: 4 extreme daily returns (>50%) — likely splits
    ✅ 10851 rows | 1980-12-15 → 2023-12-28
  Processing MSFT...
    ✅ 9525 rows | 1986-03-14 → 2023-12-28
  Processing AMZN...
  ⚠️ AMZN: 1 extreme daily returns (>50%) — likely splits
    ✅ 6699 rows | 1997-05-

In [ ]:
# =============================================================
# CELL 3: Load and clean news data (chunked for memory safety)
# =============================================================

print("Loading news data for selected tickers (chunked)...")

# Read in chunks, filter only our tickers
news_chunks = []
chunk_count = 0

for chunk in pd.read_csv(
    NEWS_FILE,
    chunksize=500_000,
    usecols=["Date", "Article_title", "Stock_symbol", "Publisher"],  # only what we need
    dtype={"Article_title": str, "Stock_symbol": str, "Publisher": str}
):
    chunk_count += 1
    # Filter to our tickers
    filtered = chunk[chunk["Stock_symbol"].isin(SELECTED_TICKERS)].copy()
    if len(filtered) > 0:
        news_chunks.append(filtered)
    print(f"  Chunk {chunk_count}: {len(chunk):,} rows → {len(filtered):,} kept")

news_raw = pd.concat(news_chunks, ignore_index=True)
print(f"\n📰 Raw news for selected tickers: {len(news_raw):,} rows")

# =============================================================
# CELL 4: Clean news data
# =============================================================

# Standardize column names
news_raw.columns = news_raw.columns.str.strip().str.lower().str.replace(" ", "_")

# Parse dates — format is "2020-06-10 07:33:26 UTC"
news_raw["datetime"] = pd.to_datetime(news_raw["date"], errors="coerce", utc=True)
news_raw = news_raw.dropna(subset=["datetime"])

# Convert to US Eastern (market timezone)
news_raw["datetime_et"] = news_raw["datetime"].dt.tz_convert("US/Eastern")

# --- Assign trading date ---
# Rule: News after 4:00 PM ET → assigned to NEXT trading day
#        News before/during market hours → assigned to THAT trading day
news_raw["hour_et"] = news_raw["datetime_et"].dt.hour
news_raw["raw_date"] = news_raw["datetime_et"].dt.date

# If published after 4 PM, shift to next calendar day
# (We'll handle weekends/holidays in the merge step)
news_raw["trade_date"] = pd.to_datetime(news_raw["raw_date"])
after_hours = news_raw["hour_et"] >= 16
news_raw.loc[after_hours, "trade_date"] += pd.Timedelta(days=1)

# --- Clean headlines ---
# Drop rows with empty or very short headlines
news_raw = news_raw.dropna(subset=["article_title"])
news_raw = news_raw[news_raw["article_title"].str.len() >= 10]  # minimum 10 chars

# Basic text cleaning (keep it light — FinBERT handles raw text well)
news_raw["headline_clean"] = (
    news_raw["article_title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)  # collapse whitespace
)

# --- Deduplicate ---
# Same headline + same ticker + same date = duplicate
before_dedup = len(news_raw)
news_raw = news_raw.drop_duplicates(subset=["headline_clean", "stock_symbol", "trade_date"])
print(f"Deduplication: {before_dedup:,} → {len(news_raw):,} ({before_dedup - len(news_raw):,} duplicates removed)")

# --- Summary ---
print(f"\n📰 Cleaned news data: {len(news_raw):,} articles")
print(f"Date range: {news_raw['trade_date'].min().date()} → {news_raw['trade_date'].max().date()}")
print(f"\nArticles per ticker:")
print(news_raw["stock_symbol"].value_counts().to_string())
print(f"\nPublisher distribution:")
print(news_raw["publisher"].value_counts().head(10).to_string())

# Keep only what we need going forward
news_clean = news_raw[["trade_date", "stock_symbol", "headline_clean", "publisher"]].copy()
news_clean = news_clean.rename(columns={"stock_symbol": "ticker"})

# Save checkpoint
news_clean.to_csv(PROCESSED_DIR / "news_clean.csv", index=False)
print(f"\n💾 Saved to {PROCESSED_DIR / 'news_clean.csv'}")

Loading news data for selected tickers (chunked)...
  Chunk 1: 500,000 rows → 10,955 kept
  Chunk 2: 500,000 rows → 10,344 kept
  Chunk 3: 500,000 rows → 3,582 kept
  Chunk 4: 500,000 rows → 0 kept
  Chunk 5: 500,000 rows → 4,528 kept
  Chunk 6: 500,000 rows → 11,319 kept
  Chunk 7: 500,000 rows → 2,647 kept
  Chunk 8: 500,000 rows → 2,165 kept
  Chunk 9: 500,000 rows → 0 kept
  Chunk 10: 500,000 rows → 0 kept
  Chunk 11: 500,000 rows → 0 kept
  Chunk 12: 500,000 rows → 0 kept
  Chunk 13: 500,000 rows → 0 kept
  Chunk 14: 500,000 rows → 0 kept
  Chunk 15: 500,000 rows → 0 kept
  Chunk 16: 500,000 rows → 0 kept
  Chunk 17: 500,000 rows → 0 kept
  Chunk 18: 500,000 rows → 0 kept
  Chunk 19: 500,000 rows → 0 kept
  Chunk 20: 500,000 rows → 0 kept
  Chunk 21: 500,000 rows → 0 kept
  Chunk 22: 500,000 rows → 0 kept
  Chunk 23: 500,000 rows → 0 kept
  Chunk 24: 500,000 rows → 0 kept
  Chunk 25: 500,000 rows → 0 kept
  Chunk 26: 500,000 rows → 0 kept
  Chunk 27: 57,514 rows → 0 kept

📰 Raw ne

In [4]:
aapl = pd.read_csv('AAPL.csv')
aapl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9338 entries, 0 to 9337
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        8865 non-null   float64
 1   Date              9338 non-null   object 
 2   Article_title     9338 non-null   object 
 3   Stock_symbol      9338 non-null   object 
 4   Url               9338 non-null   object 
 5   Publisher         473 non-null    object 
 6   Author            0 non-null      float64
 7   Article           8865 non-null   object 
 8   Lsa_summary       8865 non-null   object 
 9   Luhn_summary      8865 non-null   object 
 10  Textrank_summary  8865 non-null   object 
 11  Lexrank_summary   8865 non-null   object 
dtypes: float64(2), object(10)
memory usage: 875.6+ KB


In [8]:
sentiment = pd.read_csv('aapl_weekly_sentiment.csv')
sentiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Week           81 non-null     object 
 1   Avg_Sentiment  81 non-null     float64
 2   Article_Count  81 non-null     int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 2.0+ KB


In [9]:
sentiment.tail()

,Week,Avg_Sentiment,Article_Count
76,2023-11-13/2023-11-19,3.412986,107
77,2023-11-20/2023-11-26,3.065831,73
78,2023-11-27/2023-12-03,2.663059,96
79,2023-12-04/2023-12-10,2.235669,109
80,2023-12-11/2023-12-17,2.826030,150


,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
4473,1899.0,2009-12-02 00:00:00 UTC,Back to Basics with Steel,AA,https://www.nasdaq.com/articles/back-basics-st...,NaN,NaN,optionMONSTER submits:\nBy Guy Adami\nIn exami...,In 1907 the company was renamed Alcoa ( AA ). ...,In 1907 the company was renamed Alcoa ( AA ). ...,In 1907 the company was renamed Alcoa ( AA ). ...,In 1907 the company was renamed Alcoa ( AA ). ...
4474,1900.0,2009-11-04 00:00:00 UTC,The DJIA's Dangerous Indexing Philosophy,AA,https://www.nasdaq.com/articles/djias-dangerou...,NaN,NaN,While doing some research on the Dow Jones Ind...,While doing some research on the Dow Jones Ind...,The real problem with the index is not the top...,"The DJIA is a price weighted index, which simp...",By price I do not mean Market Capitalization b...
4475,1901.0,2009-10-14 00:00:00 UTC,The Clearly Undervalued Gem in the High-Flying...,AA,https://www.nasdaq.com/articles/clearly-underv...,NaN,NaN,Financials had a good day yesterday and the Do...,14 CloseCurrent P/EAverage 5-Year P/E2009 EPS ...,14 CloseCurrent P/EAverage 5-Year P/E2009 EPS ...,"( Alcoa ( AA ) , which has a net loss for the ...","( Alcoa ( AA ) , which has a net loss for the ..."
4476,1902.0,2009-10-09 00:00:00 UTC,"In Earnings Season, Who Cares About the Unempl...",AA,https://www.nasdaq.com/articles/earnings-seaso...,NaN,NaN,Rakesh Saxena AA\nSee also An Interview With t...,Rakesh Saxena AA See also An Interview With th...,Rakesh Saxena AA See also An Interview With th...,Rakesh Saxena AA See also An Interview With th...,Rakesh Saxena AA See also An Interview With th...
4477,1903.0,2009-10-07 00:00:00 UTC,Allow Me to Introduce: The Biggest Sucker Rall...,AA,https://www.nasdaq.com/articles/allow-me-intro...,NaN,NaN,It's been said (and perhaps you are getting ti...,Reuters reports that 'earnings optimism lift W...,Reuters reports that 'earnings optimism lift W...,Reuters reports that 'earnings optimism lift W...,Reuters reports that 'earnings optimism lift W...


In [ ]:
aapl = pd.read_csv('AAPL.csv')
aapl.info()

In [ ]:
path = '/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/full_history/'

In [ ]:
# Read the AAPL.csv file from the specified path
path = Path('/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/full_history/AAPL.csv')
aapl = pd.read_csv(path)
aapl.columns

In [ ]:
aapl.tail() 

In [12]:
import os
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION - Update these paths to match your setup
# ============================================================
INPUT_DIR = "/Users/yashaswiaryan/Projects/The WOLF/The-Wolf/full_history"    # Folder with daily CSVs
OUTPUT_DIR = "data/stock_weekly/"          # Folder for weekly CSVs

# ============================================================
# AGGREGATION RULES (Standard OHLCV weekly conversion)
# ============================================================
# Open   → First open of the week (how the week started)
# High   → Max high of the week   (peak price reached)
# Low    → Min low of the week    (lowest price reached)
# Close  → Last close of the week (how the week ended)
# Adj Close → Last adj close      (adjusted closing price)
# Volume → Sum of the week        (total shares traded)
# ============================================================

AGGREGATION_RULES = {
    'open':      'first',
    'high':      'max',
    'low':       'min',
    'close':     'last',
    'adj close': 'last',
    'volume':    'sum'
}


def convert_daily_to_weekly(filepath):
    """
    Reads a daily stock CSV and returns a weekly-aggregated DataFrame.
    
    Parameters:
        filepath (str): Path to the daily CSV file
        
    Returns:
        pd.DataFrame: Weekly aggregated stock data
    """
    # --- Step 1: Read the CSV ---
    df = pd.read_csv(filepath)
    
    # --- Step 2: Standardize column names (lowercase, strip spaces) ---
    df.columns = df.columns.str.strip().str.lower()
    
    # --- Step 3: Parse dates and sort ascending ---
    # (Your data is in descending order, so we sort ascending for correct aggregation)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # --- Step 4: Remove rows with missing critical data ---
    df = df.dropna(subset=['date', 'close'])
    
    # --- Step 5: Set date as index for resampling ---
    df = df.set_index('date')
    
    # --- Step 6: Resample to weekly frequency (W-FRI = week ending Friday) ---
    # This groups all trading days Mon-Fri into one weekly row
    weekly = df.resample('W-FRI').agg(AGGREGATION_RULES)
    
    # --- Step 7: Drop weeks with no trading data ---
    # (e.g., holiday weeks where market was closed entirely)
    weekly = weekly.dropna(subset=['close'])
    
    # --- Step 8: Round for cleaner output ---
    weekly = weekly.round(4)
    
    # --- Step 9: Reset index so 'date' becomes a column again ---
    weekly = weekly.reset_index()
    
    return weekly


def process_all_stocks(input_dir, output_dir):
    """
    Processes all daily stock CSVs in input_dir and saves weekly versions to output_dir.
    
    Parameters:
        input_dir  (str): Folder containing daily stock CSVs (e.g., AAPL.csv)
        output_dir (str): Folder to save weekly aggregated CSVs
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all CSV files
    csv_files = sorted(Path(input_dir).glob("*.csv"))
    
    if not csv_files:
        print(f"⚠️  No CSV files found in: {input_dir}")
        return
    
    print(f"📂 Found {len(csv_files)} stock files to process\n")
    
    success_count = 0
    error_files = []
    summary_rows = []
    
    for filepath in csv_files:
        ticker = filepath.stem  # e.g., "AAPL" from "AAPL.csv"
        
        try:
            weekly_df = convert_daily_to_weekly(filepath)
            
            # Save to output directory
            output_path = os.path.join(output_dir, f"{ticker}.csv")
            weekly_df.to_csv(output_path, index=False)
            
            # Track stats
            daily_count = len(pd.read_csv(filepath))
            weekly_count = len(weekly_df)
            date_range = f"{weekly_df['date'].min().date()} → {weekly_df['date'].max().date()}"
            
            summary_rows.append({
                'ticker': ticker,
                'daily_rows': daily_count,
                'weekly_rows': weekly_count,
                'date_range': date_range
            })
            
            success_count += 1
            print(f"  ✅ {ticker:6s} | {daily_count:>6} daily → {weekly_count:>5} weekly | {date_range}")
            
        except Exception as e:
            error_files.append((ticker, str(e)))
            print(f"  ❌ {ticker:6s} | Error: {e}")
    
    # --- Final Summary ---
    print(f"\n{'='*60}")
    print(f"📊 Processing Complete!")
    print(f"   ✅ Success: {success_count}/{len(csv_files)}")
    if error_files:
        print(f"   ❌ Errors:  {len(error_files)}")
        for ticker, err in error_files:
            print(f"      - {ticker}: {err}")
    print(f"   📁 Output:  {output_dir}")
    print(f"{'='*60}")
    
    # Optional: Save summary as CSV
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        summary_path = os.path.join(output_dir, "_processing_summary.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"\n📋 Summary saved to: {summary_path}")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    process_all_stocks(INPUT_DIR, OUTPUT_DIR)

  ✅ EGY    |   7785 daily →  1614 weekly | 1993-01-29 → 2023-12-29
  ✅ EH     |   1018 daily →   212 weekly | 2019-12-13 → 2023-12-29
  ✅ EHC    |   9391 daily →  1945 weekly | 1986-09-26 → 2023-12-29
  ✅ EHI    |   5141 daily →  1066 weekly | 2003-08-01 → 2023-12-29
  ✅ EHTH   |   4326 daily →   898 weekly | 2006-10-20 → 2023-12-29
  ✅ EIC    |   3818 daily →   860 weekly | 2007-06-01 → 2023-12-29
  ✅ EIDO   |   3435 daily →   713 weekly | 2010-05-07 → 2023-12-29
  ✅ EIDX   |    513 daily →   107 weekly | 2018-06-22 → 2020-07-03
  ✅ EIG    |   4258 daily →   883 weekly | 2007-02-02 → 2023-12-29
  ✅ EIGI   |   1619 daily →   337 weekly | 2013-10-25 → 2020-04-03
  ✅ EIGR   |   2496 daily →   518 weekly | 2014-01-31 → 2023-12-29
  ✅ EIM    |   5371 daily →  1114 weekly | 2002-08-30 → 2023-12-29
  ✅ EINC   |   2968 daily →   616 weekly | 2012-03-16 → 2023-12-29
  ✅ EIS    |   3967 daily →   823 weekly | 2008-03-28 → 2023-12-29
  ✅ EIX    |  12776 daily →  2644 weekly | 1973-05-04 → 2023-1

KeyboardInterrupt: 